In [0]:
use catalog identifier(:catalog);
use schema identifier(:schema);

In [0]:
declare or replace qry_str string;

set var qry_str = 
  "create or replace view metv_query_history
  with metrics
  language yaml
  as $$
  version: 1.1
  comment: DBSQL query performance metrics by warehouse and date.
  
  source: " || :catalog || '.' || :schema || ".fct_query_history
  
  joins:
    - name: calendar
      source: " || :catalog || '.' || :schema || ".dim_calendar
      on: calendar.calendar_key = source.calendar_key

    - name: compute
      source: " || :catalog || '.' || :schema || ".dim_compute
      on: compute.compute_key = source.compute_key

  
  dimensions:
    - name: Date
      expr: calendar.calendar_date
      display_name: Query Date
      comment: Date when Databricks received the request.
      synonyms:
        - run date
        - execution date
        - statement date
    - name: Client Application
      expr: source.client_application
      display_name: Client App
      comment: 'Client application that ran the statement. For example: Databricks SQL, Tableau, and Power BI. Nulls and blanks are labeled as Unknown.'
      synonyms:
        - app
        - application
        - tool
        - BI tool
    - name: Warehouse Name
      expr: compute.warehouse_name
      display_name: Warehouse
      comment: The name of the SQL warehouse.
      synonyms:
        - warehouse
        - endpoint
        - SQL warehouse
    - name: Warehouse Type
      expr: compute.warehouse_type
      display_name: Warehouse Type
      comment: The type of SQL warehouse. Possible values are CLASSIC, PRO, and SERVERLESS.
      synonyms:
        - warehouse tier
        - endpoint type
    - name: Warehouse Size
      expr: compute.warehouse_size
      display_name: Warehouse Size
      comment: The cluster size of the SQL warehouse. Possible values are 2X_SMALL, X_SMALL, SMALL, MEDIUM, LARGE, X_LARGE, 2X_LARGE, 3X_LARGE, and 4X_LARGE.
      synonyms:
        - cluster size
        - t-shirt size

  measures:
    - name: Query Count
      expr: count(statement_id)
      display_name: Total Queries
      comment: Count of queries.
      synonyms:
        - number of queries
        - statement count
        - query volume
    - name: Query Duration Sec P50
      expr: percentile(total_duration_ms / 1000, 0.5)
      display_name: Median Duration (s)
      comment: P50 of total execution time of the statement in seconds (Excluding result fetch time).
      synonyms:
        - median latency
        - p50 duration
        - typical query time
    - name: Query Duration Sec P90
      expr: percentile(total_duration_ms / 1000, 0.9)
      display_name: P90 Duration (s)
      comment: P90 of total execution time of the statement in seconds (Excluding result fetch time).
      synonyms:
        - p90 latency
        - 90th percentile duration
    - name: Query Duration Sec P95
      expr: percentile(total_duration_ms / 1000, 0.95)
      display_name: P95 Duration (s)
      comment: P95 of total execution time of the statement in seconds (Excluding result fetch time).
      synonyms:
        - p95 latency
        - 95th percentile duration
        - tail latency
  $$";

execute immediate qry_str;